In [ ]:
"""
TBD: clean up
"""

In [ ]:
import os
import numpy as np
import pandas as pd
from numpy import ma
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from src import mobi

In [ ]:
general_dir = "../../result/UnifyParaSearch/fly/"
motif_dir = general_dir+"/inference_motif/"
tomtom_dir = general_dir+"/inference_tomtom/"
spp_prefix = "DREME_RankSPP_100_"
mobi_prefix = "DREME_RankLinear_6.0_20_"
mockIP_motif_dir = "../../result/mockIP/fly/inference_motif/"
mockIP_tomtom_dir = "../../result/mockIP/fly/inference_tomtom/"

In [ ]:
# all TFs
tfs = [i for i in os.listdir(tomtom_dir) if i.startswith(spp_prefix) and not i.endswith("/")]
tfs = [i.split("_")[3] for i in tfs]
tfs = [i.replace(".tomtom", "") for i in tfs]
tfs = np.unique(tfs)

# TFs in mockIP
mock_tfs = [i for i in os.listdir("../../result/mockIP/fly/inference_motif/") if i.startswith('DREME_RankSPP_20_')]
mock_tfs = [i.split("_")[3] for i in mock_tfs]
mock_tfs = [i.replace(".meme", "") for i in mock_tfs]

tfs = list(set.intersection(set(mock_tfs), set(tfs)))

In [ ]:
coverage = []
accuracy = []
spp_overlap = []
mock_spp_overlap = []
mobi_spp_overlap = []

spp_result = []
mock_result = []
mobi_result = []

result = []

for i in range(len(tfs)):
    
    #### combine spp and mobi ####
    # get spp motif arrays
    n2i = get_inference_motif_order(motif_dir+spp_prefix+tfs[i]+".meme")
    n2a = get_n2a(motif_dir+spp_prefix+tfs[i]+".meme")
    i2n = {}
    for j in n2i.keys():
        i2n[n2i[j]] = j
    motif_spp = [n2a[i2n[j]].reshape(-1) for j in range(5)]

    # get mobi motif arrays
    n2i = get_inference_motif_order(motif_dir+mobi_prefix+tfs[i]+".meme")
    n2a = get_n2a(motif_dir+mobi_prefix+tfs[i]+".meme")
    i2n = {}
    for j in n2i.keys():
        i2n[n2i[j]] = j
    motif_mobi = [n2a[i2n[j]].reshape(-1) for j in range(5)]

    # combine 10 motifs, mobi first
    motifs = np.array(motif_mobi + motif_spp)

    # k-means
    km = KMeans(n_clusters=5, random_state=12345)
    km.fit(motifs)

    # index of 10 motifs that should be picked
    motif_ind = []
    for ii in range(5):
        pick = np.argwhere(km.labels_ == ii).reshape(-1)
        if(len(pick) == 1):
            motif_ind.append(pick[0])
        else:
            pick_copy = pick.copy()
            pick_copy[pick>4] = pick[pick>4]-5
            motif_ind.append(pick[pick_copy.argmin()])
    motif_ind = np.sort(motif_ind)
    mobi_ind = motif_ind[motif_ind<5]
    spp_ind = motif_ind[motif_ind>=5] - 5
    
    
    # mobi part
    seq2ind = get_inference_motif_order(motif_dir+mobi_prefix+tfs[i]+".meme")
    mobi = pd.read_csv(tomtom_dir+mobi_prefix+tfs[i]+".tomtom", sep="\t")
    mobi = mobi[mobi['p-value'] <= 0.05] # significant
    mobi = mobi.sort_values('p-value').drop_duplicates("#Query ID") # for one predict motif, only use the best matched known motif
    match_ind = [seq2ind[jj] for jj in mobi["#Query ID"].values]
    mobi_ind_tomtom = [jj for jj in range(len(match_ind)) if match_ind[jj] in mobi_ind] # only keep predict motif that is in the first n or in the mobi_ind
    mobi = mobi.iloc[mobi_ind_tomtom,:].copy() # final matched df

    
    # combine
    combine = pd.concat([mobi, spp])
    #### combine spp and mobi end ####
    
    # spp only
    seq2ind = get_inference_motif_order(motif_dir+spp_prefix+tfs[i]+".meme")
    spp = pd.read_csv(tomtom_dir+spp_prefix+tfs[i]+".tomtom", sep="\t")
    spp = spp[spp['p-value'] <= 0.05] # significant
    spp = spp.sort_values('p-value').drop_duplicates("#Query ID") # for one predict motif, only use the best matched known motif
    match_ind = [seq2ind[jj] for jj in spp["#Query ID"].values]
    spp_ind_tomtom = [jj for jj in range(len(match_ind)) if match_ind[jj] in [0,1,2,3,4]] # only keep predict motif that is in the first n or in the spp_ind
    spp = spp.iloc[spp_ind_tomtom,:].copy() # final matched df
    
    # mobi only
    seq2ind = get_inference_motif_order(motif_dir+mobi_prefix+tfs[i]+".meme")
    mobi = pd.read_csv(tomtom_dir+mobi_prefix+tfs[i]+".tomtom", sep="\t")
    mobi = mobi[mobi['p-value'] <= 0.05] # significant
    mobi = mobi.sort_values('p-value').drop_duplicates("#Query ID") # for one predict motif, only use the best matched known motif
    match_ind = [seq2ind[jj] for jj in mobi["#Query ID"].values]
    mobi_ind_tomtom = [jj for jj in range(len(match_ind)) if match_ind[jj] in [0,1,2,3,4]] # only keep predict motif that is in the first n or in the mobi_ind
    mobi = mobi.iloc[mobi_ind_tomtom,:].copy() # final matched df

    # mockIP only
    seq2ind = get_inference_motif_order(mockIP_motif_dir+spp_prefix+tfs[i]+".meme")
    mock = pd.read_csv(mockIP_tomtom_dir+spp_prefix+tfs[i]+".tomtom", sep="\t")
    mock = mock[mock['p-value'] <= 0.05] # significant
    mock = mock.sort_values('p-value').drop_duplicates("#Query ID") # for one predict motif, only use the best matched known motif
    match_ind = [seq2ind[jj] for jj in mock["#Query ID"].values]
    mock_ind_tomtom = [jj for jj in range(len(match_ind)) if match_ind[jj] in [0,1,2,3,4]] # only keep predict motif that is in the first n or in the mock_ind
    mock = mock.iloc[mock_ind_tomtom,:].copy() # final matched df

    
    
    
    ## report stats
    
#     for df_i in [spp, mobi, mock]: #combine, mock]:

# #         # coverage
# #         coverage.append(len(df_i["Target ID"].unique()))
# #         # accuracy
# #         accuracy.append(df_i.shape[0])
#         # known motif coverage / spp known motif coverage
#         n_overlap = len(set.intersection(set(spp["Target ID"].unique()), set(df_i["Target ID"].unique())))
#         n_spp = len(spp["Target ID"].unique())
#         if n_spp != 0:
#             spp_overlap.append(n_overlap/n_spp)
#         else:
#             spp_overlap.append(np.nan)
#         # known motif coverage / mockIP known motif coverage
#         n_overlap = len(set.intersection(set(mock["Target ID"].unique()), set(df_i["Target ID"].unique())))
#         n_spp = len(mock["Target ID"].unique())
#         if n_spp != 0:
#             mock_spp_overlap.append(n_overlap/n_spp)
#         else:
#             mock_spp_overlap.append(np.nan)
            
#         n_overlap = len(set.intersection(set(mobi["Target ID"].unique()), set(df_i["Target ID"].unique())))
#         n_spp = len(mobi["Target ID"].unique())
#         if n_spp != 0:
#             mobi_spp_overlap.append(n_overlap/n_spp)
#         else:
#             mobi_spp_overlap.append(np.nan)

        # known motif coverage / mockIP known motif coverage
    spp_overlap = set.intersection(set(mock["Target ID"].unique()), set(spp["Target ID"].unique()))
    mobi_overlap = set.intersection(set(mock["Target ID"].unique()), set(mobi["Target ID"].unique()))

    double_overlap = set.intersection(spp_overlap, mobi_overlap)
    
    if len(spp_overlap) != 0:
        ff = len(double_overlap)/len(spp_overlap)
        result.append(ff)
    else:
        result.append(np.nan)
    if len(mobi_overlap) != 0:
        ff2 = len(double_overlap)/len(mobi_overlap)
        result.append(ff2)
    else:
        result.append(np.nan)


#     # all sig-hit motifs entropy
#     mock_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/mockIP/fly/inference_motif/DREME_RankSPP_20_%s.meme" % (tfs[i]))
#     spp_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/UnifyParaSearch/fly/inference_motif/DREME_RankSPP_20_%s.meme" % (tfs[i]))
#     mobi_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/UnifyParaSearch/fly/inference_motif/DREME_RankLinear_6.0_20_%s.meme" % (tfs[i]))
    
#     spp_entropy = [motif_entropy(spp_n2a[ii]) for ii in spp["#Query ID"].values]
#     mock_entropy = [motif_entropy(mock_n2a[ii]) for ii in mock["#Query ID"].values]
#     mobi_entropy = [motif_entropy(mobi_n2a[ii]) for ii in mobi["#Query ID"].values]
    
#     if spp_entropy:
#         spp_result.append(np.mean(spp_entropy))
#     else:
#         spp_result.append(np.nan)
#     if mock_entropy:
#         mock_result.append(np.mean(mock_entropy))
#     else:
#         mock_result.append(np.nan)
#     if mobi_entropy:
#         mobi_result.append(np.mean(mobi_entropy))
#     else:
#         mobi_result.append(np.nan)        
      
      # unique and overlap entropy
#     spp_unique = spp[~spp['Target ID'].isin(mock["Target ID"].values)]
#     mock_unique = mock[~mock['Target ID'].isin(spp["Target ID"].values)]
#     mock_u_motifs = mock_unique["#Query ID"].values
#     spp_u_motifs = spp_unique["#Query ID"].values

#     spp_overlap = spp[spp['Target ID'].isin(mock["Target ID"].values)]
#     mock_overlap = mock[mock['Target ID'].isin(spp["Target ID"].values)]
#     mock_o_motifs = mock_overlap["#Query ID"].values
#     spp_o_motifs = spp_overlap["#Query ID"].values

#     mock_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/mockIP/fly/inference_motif/DREME_RankSPP_20_%s.meme" % (tfs[i]))
#     spp_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/UnifyParaSearch/fly/inference_motif/DREME_RankSPP_20_%s.meme" % (tfs[i]))

#     spp_entropy = [motif_entropy(spp_n2a[ii]) for ii in spp_o_motifs]
#     mock_entropy = [motif_entropy(mock_n2a[ii]) for ii in mock_o_motifs]
#     overlap_entropy = mock_entropy+spp_entropy

#     spp_entropy = [motif_entropy(spp_n2a[ii]) for ii in spp_u_motifs]
#     mock_entropy = [motif_entropy(mock_n2a[ii]) for ii in mock_u_motifs]

#     if spp_entropy:
#         spp_result.append(np.mean(spp_entropy))
#     else:
#         spp_result.append(np.nan)
#     if mock_entropy:
#         mock_result.append(np.mean(mock_entropy))
#     else:
#         mock_result.append(np.nan)
#     if overlap_entropy:
#         overlap_result.append(np.mean(overlap_entropy))
#     else:
#         overlap_result.append(np.nan)

#     # all 5 inferred motif entropy
#     mock_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/mockIP/fly/inference_motif/DREME_RankSPP_100_%s.meme" % (tfs[i]))
#     spp_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/UnifyParaSearch/fly/inference_motif/DREME_RankSPP_100_%s.meme" % (tfs[i]))
#     mobi_n2a = get_n2a("/gpfs/slayman/pi/gerstein/jg2447/motif_inference/result/UnifyParaSearch/fly/inference_motif/DREME_RankLinear_6.0_20_%s.meme" % (tfs[i]))

#     n2i = get_inference_motif_order(motif_dir+spp_prefix+tfs[i]+".meme")        
#     spp_motifs = [ii for ii in n2i.keys() if n2i[ii] < 5]
#     n2i = get_inference_motif_order(motif_dir+mobi_prefix+tfs[i]+".meme")        
#     mobi_motifs = [ii for ii in n2i.keys() if n2i[ii] < 5]
#     n2i = get_inference_motif_order(mockIP_motif_dir+spp_prefix+tfs[i]+".meme")        
#     mock_motifs = [ii for ii in n2i.keys() if n2i[ii] < 5]
    
#     spp_entropy = [motif_entropy(spp_n2a[ii]) for ii in spp_motifs]
#     mock_entropy = [motif_entropy(mock_n2a[ii]) for ii in mock_motifs]
#     mobi_entropy = [motif_entropy(mobi_n2a[ii]) for ii in mobi_motifs]
    
#     if spp_entropy:
#         spp_result.append(np.mean(spp_entropy))
#     else:
#         spp_result.append(np.nan)
#     if mock_entropy:
#         mock_result.append(np.mean(mock_entropy))
#     else:
#         mock_result.append(np.nan)
#     if mobi_entropy:
#         mobi_result.append(np.mean(mobi_entropy))
#     else:
#         mobi_result.append(np.nan)  

In [ ]:
# all TFs
tfs = [i for i in os.listdir("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/fly/inference_motif/") if i.startswith(spp_prefix) and not i.endswith("/")]
tfs = [i.split("_")[3] for i in tfs]
tfs = [i.replace(".meme", "") for i in tfs]
tfs = np.unique(tfs)

In [ ]:
# all TFs
tfs2 = [i for i in os.listdir("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch_nSeq500/fly/inference_motif/") if i.startswith("DREME_RankSPP_100_") and not i.endswith("/")]
tfs2 = [i.split("_")[3] for i in tfs2]
tfs2 = [i.replace(".meme", "") for i in tfs2]
tfs2 = np.unique(tfs2)

In [ ]:
tfs = list(set.intersection(set(tfs2), set(tfs)))

In [ ]:
def get_n2a(mfile):
    motifs = []
    motif_names = []
    with open(mfile, "r") as f:
        lines = f.readlines()
        for k in range(len(lines)):
            if lines[k].startswith("letter-probability"):
                j = k
                while True:
                    j = j+1
                    if j < len(lines):
                        if lines[j].startswith(" 0.") or lines[j].startswith(" 1."):
                            pass
                        else:
                            break
                    else:
                        break
                motif = np.array(lines)[k+1:j]
                motif_array = []
                for i in motif:
                    motif_array.append(i.rstrip().split("\t"))
                motif = np.array(motif_array).astype(float)
                motifs.append(motif)
            if lines[k].startswith("MOTIF"):
                name = lines[k].split(" ")[1].rstrip()
                motif_names.append(name)
    n2a = {}
    for i,j in zip(motifs, motif_names):
        n2a[j] = i
    return(n2a)

In [ ]:
result = []
for tf in tfs:
    n2a = get_n2a8("/home/jg2447/slayman/data/motif_cisbp/meme_fly_stack/%s.meme" % tf)
    for ii in n2a.keys():
        result.append(motif_entropy(n2a[ii]))

In [ ]:
plt.hist(result,bins=65,range=(0,13))
plt.xlabel("entropy")